<a href="https://colab.research.google.com/github/BlackBossX/Brain-Tumor-Classification/blob/main/notebooks/BrainTumorClassification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [36]:
import pandas as pd
import numpy as np

#ML
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier


import tensorflow as tf
from tensorflow import keras


import matplotlib.pyplot as plt
import seaborn as sns

# Model Saving
import joblib


print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Scikit-learn:", __import__("sklearn").__version__)
print("TensorFlow:", tf.__version__)
print("Matplotlib:", plt.matplotlib.__version__)
print("Seaborn:", sns.__version__)

Pandas: 2.2.2
NumPy: 2.0.2
Scikit-learn: 1.6.1
TensorFlow: 2.20.0
Matplotlib: 3.10.0
Seaborn: 0.13.2


In [37]:
df = pd.read_csv("/content/drive/MyDrive/BrainTumorClassification/brain_tumor_data.csv");

In [39]:
df.isna().sum()

,0
patient_id,0
age,0
gender,0
ethnicity,281
region,0
bmi,436
smoking_status,381
alcohol_consumption,3804
family_history,356
tumor_size_mm,0


In [40]:
df.head()

,patient_id,age,gender,ethnicity,region,bmi,smoking_status,alcohol_consumption,family_history,tumor_size_mm,...,ct_density,edema_grade,contrast_enhancement,ki67_index,bp_systolic,bp_diastolic,wbc_count,crp_level,genetic_marker_status,tumor_type
0,PT100001,60,Male,African,Rural,28.1,Never,Moderate,No,13.0,...,51.4,0.0,Mild,0.7,103,85.0,7.56,7.81,Positive,Meningioma
1,PT100002,50,Male,Asian,Urban,25.0,Never,Moderate,No,14.5,...,35.2,0.0,NaN,NaN,134,97.0,9.17,4.37,Positive,Pituitary
2,PT100003,74,Female,Other,Suburban,21.0,Never,NaN,No,10.7,...,43.0,1.0,Strong,3.4,127,62.0,3.67,10.16,Inconclusive,Meningioma
3,PT100004,34,Male,Hispanic,Suburban,29.2,Never,NaN,No,36.7,...,18.5,3.0,Mild,5.6,100,80.0,7.55,1.70,Positive,Glioma
4,PT100005,59,Male,Asian,Rural,28.2,Former,Moderate,No,16.0,...,24.0,1.0,Moderate,1.4,132,77.0,9.35,8.14,?,Pituitary


In [41]:
df.shape
#df.dtypes

(9000, 29)

In [42]:
#remove patient_id column cz not necessary for our predictions
df = df.drop("patient_id",axis=1)

In [43]:
df.head()

,age,gender,ethnicity,region,bmi,smoking_status,alcohol_consumption,family_history,tumor_size_mm,tumor_location,...,ct_density,edema_grade,contrast_enhancement,ki67_index,bp_systolic,bp_diastolic,wbc_count,crp_level,genetic_marker_status,tumor_type
0,60,Male,African,Rural,28.1,Never,Moderate,No,13.0,Cerebellum,...,51.4,0.0,Mild,0.7,103,85.0,7.56,7.81,Positive,Meningioma
1,50,Male,Asian,Urban,25.0,Never,Moderate,No,14.5,Sellar,...,35.2,0.0,NaN,NaN,134,97.0,9.17,4.37,Positive,Pituitary
2,74,Female,Other,Suburban,21.0,Never,NaN,No,10.7,Temporal,...,43.0,1.0,Strong,3.4,127,62.0,3.67,10.16,Inconclusive,Meningioma
3,34,Male,Hispanic,Suburban,29.2,Never,NaN,No,36.7,Occipital,...,18.5,3.0,Mild,5.6,100,80.0,7.55,1.70,Positive,Glioma
4,59,Male,Asian,Rural,28.2,Former,Moderate,No,16.0,Sellar,...,24.0,1.0,Moderate,1.4,132,77.0,9.35,8.14,?,Pituitary


In [44]:
df['alcohol_consumption'] = df['alcohol_consumption'].replace('Unknown', np.nan)
df['genetic_marker_status'] = df['genetic_marker_status'].replace('?', np.nan)


In [45]:
df.alcohol_consumption
df.genetic_marker_status

,genetic_marker_status
0,Positive
1,Positive
2,Inconclusive
3,Positive
4,NaN
...,...
8995,Positive
8996,Positive
8997,Negative
8998,Positive


In [46]:
df.isna().sum()

,0
age,0
gender,0
ethnicity,281
region,0
bmi,436
smoking_status,381
alcohol_consumption,4185
family_history,356
tumor_size_mm,0
tumor_location,0


In [47]:
df.dtypes

,0
age,int64
gender,object
ethnicity,object
region,object
bmi,float64
smoking_status,object
alcohol_consumption,object
family_history,object
tumor_size_mm,float64
tumor_location,object


In [48]:
x = df.drop(columns='tumor_type')
y = df['tumor_type']


In [57]:
x.head()
y.head()

,tumor_type
0,Meningioma
1,Pituitary
2,Meningioma
3,Glioma
4,Pituitary


In [59]:
df['tumor_type'].value_counts()

,count
tumor_type,
Glioma,3802
Meningioma,3089
Pituitary,2109


In [60]:
X_train , X_temp , Y_train , Y_temp = train_test_split(
    x,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
    )

In [62]:
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    Y_temp,
    test_size=0.50,
    random_state=42,
    stratify=Y_temp
)